# Kaggle Training: LogFilter NER on Log Corpus

Fine-tunes `cisco-ai/SecureBERT2.0-NER` on log-domain text. The published variant is pre-trained on cyber-threat-intelligence prose (threat reports, CVE summaries) and recognises five entity types — `Indicator`, `Malware`, `Organization`, `System`, `Vulnerability`. Production logs contain the same entity *kinds* in different syntactic forms (e.g. `src: /10.251.74.62:35977` instead of `the actor used 10.251.74.62`). Without log-adapted fine-tuning, NER recall drops on log syntax, which weakens the `entity_boost` term in `LogScorer` and the `ai_entities` LEEF field.

This notebook uses **regex weak supervision** to bootstrap labelled spans (IPs, hashes, CVEs, file paths, etc.) directly from the HDFS TraceBench reconstructed corpus, then fine-tunes the token classifier on those weak labels. This trades label noise for label volume — the right tradeoff when a hand-labelled log NER corpus does not exist.

Output: `models/ner/` — HuggingFace token-classification directory drop-in for `src/logfilter/models/ner.py`. Expected runtime on Kaggle Free GPU (T4) is well under the 9h session limit. Enable a GPU accelerator before running training cells.

## 1. Locate the repo

Run this notebook from the project root when possible. If Kaggle places the repo inside `/kaggle/working`, the cell below tries to find it.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()
GIT_URL = 'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git'

if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]
    else:
        clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
        if not clone_dir.exists():
            subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(clone_dir)], check=True)
        REPO_DIR = clone_dir

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo into /kaggle/working first.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)

## 2. Install training dependencies

Adds `seqeval` for token-level F1, plus the standard transformers/datasets/optimum stack.

In [ ]:
%pip install -q torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
%pip install -q --upgrade-strategy only-if-needed datasets evaluate seqeval optimum[onnxruntime]

## 3. Attach the HDFS TraceBench dataset

Same dataset wiring as the other Kaggle notebooks. The reconstructed text windows are the unlabelled raw input the regex weak supervision will run over.

In [ ]:
PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'

def has_trace_files(path: Path) -> bool:
    return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

if not has_trace_files(PREPROCESSED_DIR):
    input_root = Path('/kaggle/input')
    candidates = [p for p in input_root.rglob('preprocessed') if has_trace_files(p)]
    if not candidates:
        # Datasets may publish trace files at the top level (no nested 'preprocessed' dir).
        candidates = [p for p in input_root.rglob('*') if p.is_dir() and has_trace_files(p)]
    if not candidates:
        raise FileNotFoundError('No Kaggle input folder with normal_trace.csv and failure_trace.csv was found.')
    source = candidates[0]
    PREPROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PREPROCESSED_DIR.exists() and not PREPROCESSED_DIR.is_symlink():
        raise FileExistsError(f'{PREPROCESSED_DIR} exists but does not contain the expected trace files.')
    if not PREPROCESSED_DIR.exists():
        os.symlink(source, PREPROCESSED_DIR, target_is_directory=True)
    print('Linked dataset:', source, '->', PREPROCESSED_DIR)
else:
    print('Dataset already available:', PREPROCESSED_DIR)

## 4. Weak supervision — regex labelling

The patterns below cover the entity surface forms that actually occur in syslog/system-log text. Each match becomes a labelled span. After labelling, we tokenise with the SecureBERT2.0 tokenizer's offset-mapping and project the character-level spans onto sub-word tokens using the standard B-/I- BIO scheme (the first token of an entity gets `B-`, subsequent tokens get `I-`, everything else is `O`).

In [ ]:
import re
from training.text_dataset import build_windows

LABEL_LIST = [
    'O',
    'B-Indicator', 'I-Indicator',
    'B-Malware', 'I-Malware',
    'B-Organization', 'I-Organization',
    'B-System', 'I-System',
    'B-Vulnerability', 'I-Vulnerability',
]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

# Patterns: (compiled regex, entity type). Order matters when patterns overlap;
# we keep the longest non-overlapping match per text (resolved below).
PATTERNS = [
    (re.compile(r'\bCVE-\d{4}-\d{4,7}\b', re.IGNORECASE), 'Vulnerability'),
    (re.compile(r'\b[a-fA-F0-9]{64}\b'), 'Indicator'),                      # SHA-256
    (re.compile(r'\b[a-fA-F0-9]{40}\b'), 'Indicator'),                      # SHA-1
    (re.compile(r'\b[a-fA-F0-9]{32}\b'), 'Indicator'),                      # MD5
    (re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}(?::\d{1,5})?\b'), 'Indicator'),  # IPv4 [+ optional port]
    (re.compile(r'\b(?:[A-Fa-f0-9]{1,4}:){2,7}[A-Fa-f0-9]{1,4}\b'), 'Indicator'),  # IPv6 (loose)
    (re.compile(r'\bhttps?://[^\s"\'<>]+', re.IGNORECASE), 'Indicator'),    # URL
    (re.compile(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b'), 'Indicator'),  # email
    (re.compile(r'(?:/[A-Za-z0-9._\-]+){2,}'), 'System'),                   # POSIX path
    (re.compile(r'\b[A-Z]:\\(?:[^\\\s"]+\\?)+', re.IGNORECASE), 'System'),   # Windows path
    (re.compile(r'\b[A-Za-z0-9_-]+\.(?:exe|dll|so|jar|sh|bat|ps1|py)\b', re.IGNORECASE), 'System'),  # executables
]

_DOMAIN_STOPWORDS = {'java.io', 'java.lang', 'java.util', 'java.net', 'sun.misc'}
_DOMAIN_RE = re.compile(r'\b(?:[a-z0-9](?:[a-z0-9-]*[a-z0-9])?\.)+[a-z]{2,}\b', re.IGNORECASE)

def find_spans(text: str) -> list[tuple[int, int, str]]:
    spans: list[tuple[int, int, str]] = []
    for pat, ent in PATTERNS:
        for m in pat.finditer(text):
            spans.append((m.start(), m.end(), ent))
    # Domains last + with stopword filter (avoid labelling Java package paths)
    for m in _DOMAIN_RE.finditer(text):
        token = m.group(0).lower()
        if token in _DOMAIN_STOPWORDS:
            continue
        if any(token.startswith(p) for p in _DOMAIN_STOPWORDS):
            continue
        spans.append((m.start(), m.end(), 'Indicator'))
    if not spans:
        return []
    # Resolve overlaps: prefer earliest start, longer length wins ties
    spans.sort(key=lambda s: (s[0], -(s[1] - s[0])))
    keep: list[tuple[int, int, str]] = []
    last_end = -1
    for s, e, ent in spans:
        if s >= last_end:
            keep.append((s, e, ent))
            last_end = e
    return keep

# Sanity-check: label one preview window
preview_texts, _, _ = build_windows(sample_normal=1, sample_failure=1, random_state=42)
for i, t in enumerate(preview_texts):
    spans = find_spans(t)
    print(f'\n--- preview {i}, {len(spans)} weak labels ---')
    for s, e, ent in spans[:12]:
        print(f'  [{ent:13}] "{t[s:e]}"')
    if len(spans) > 12:
        print(f'  ... [+{len(spans) - 12} more]')

## 5. Sampled NER training run (verify environment first)

Build a small (5K normal + 5K failure) labelled-via-regex dataset, tokenise with offset mapping, project spans to BIO sub-word labels, and run two epochs of token-classification fine-tuning on `cisco-ai/SecureBERT2.0-NER`. Output: `models/ner-sample/`.

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

# To consume the MLM-adapted base produced by kaggle_pretrain_mlm.ipynb,
# replace MODEL_ID with 'models/securebert2-logs-mlm/final' and reinitialise
# the classification head (HF will warn that the head weights are not loaded;
# that is intended).
MODEL_ID = 'cisco-ai/SecureBERT2.0-NER'
MAX_LENGTH = 512

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

print('CUDA:', torch.cuda.is_available(), 'bf16:', use_bf16, 'fp16:', use_fp16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

sample_texts, _, _ = build_windows(sample_normal=5000, sample_failure=5000, random_state=42)

# Tokenise once with offsets and project regex spans into per-token labels.
def encode(text: str) -> dict:
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )
    offsets = enc.pop('offset_mapping')
    spans = find_spans(text)
    labels = [LABEL2ID['O']] * len(offsets)
    for s, e, ent in spans:
        first = True
        for tok_idx, (tok_s, tok_e) in enumerate(offsets):
            if tok_s == 0 and tok_e == 0:  # special tokens
                labels[tok_idx] = -100
                continue
            if tok_e <= s or tok_s >= e:
                continue
            tag = f'B-{ent}' if first else f'I-{ent}'
            labels[tok_idx] = LABEL2ID[tag]
            first = False
    # Also mask special tokens that never overlapped any span
    for tok_idx, (tok_s, tok_e) in enumerate(offsets):
        if tok_s == 0 and tok_e == 0 and labels[tok_idx] == LABEL2ID['O']:
            labels[tok_idx] = -100
    enc['labels'] = labels
    return enc

encoded = [encode(t) for t in sample_texts]
ds = Dataset.from_list(encoded)
ds_split = ds.train_test_split(test_size=0.1, seed=42)
print('train/eval:', len(ds_split['train']), '/', len(ds_split['test']))

# Distribution sanity check
from collections import Counter
label_counts = Counter()
for ex in ds_split['train']:
    label_counts.update(l for l in ex['labels'] if l != -100)
print('label counts (train):', {ID2LABEL[k]: v for k, v in label_counts.most_common()})

collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

import evaluate as _evaluate  # seqeval-backed via 'evaluate'
try:
    metric = _evaluate.load('seqeval')
except Exception:
    metric = None

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=-1)
    true_labels = [
        [ID2LABEL[l] for (p, l) in zip(pp, ll) if l != -100]
        for pp, ll in zip(preds, labels)
    ]
    true_preds = [
        [ID2LABEL[p] for (p, l) in zip(pp, ll) if l != -100]
        for pp, ll in zip(preds, labels)
    ]
    if metric is None:
        return {}
    res = metric.compute(predictions=true_preds, references=true_labels, zero_division=0)
    return {
        'precision': res['overall_precision'],
        'recall': res['overall_recall'],
        'f1': res['overall_f1'],
        'accuracy': res['overall_accuracy'],
    }

sample_out = REPO_DIR / 'models' / 'ner-sample'
args = TrainingArguments(
    output_dir=str(sample_out),
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=use_bf16,
    fp16=use_fp16,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=50,
    save_total_limit=1,
    report_to='none',
    dataloader_num_workers=2,
    seed=42,
    metric_for_best_model='f1',
    load_best_model_at_end=True,
)
trainer = Trainer(
    model=model,
    args=args,
    data_collator=collator,
    train_dataset=ds_split['train'],
    eval_dataset=ds_split['test'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()
metrics = trainer.evaluate()
print('sampled eval:', metrics)

trainer.save_model(str(sample_out / 'final'))
tokenizer.save_pretrained(str(sample_out / 'final'))
(sample_out / 'final' / 'ner_metrics.json').write_text(json.dumps(metrics, indent=2))
(sample_out / 'final' / 'ner_label_map.json').write_text(json.dumps(ID2LABEL, indent=2))
print('saved:', sample_out / 'final')

## 6. Full NER training run (uncomment when sampled run succeeds)

Use all reconstructed windows (~226K) for two epochs. Output: `models/ner/final/`. Estimated runtime ~2-4h on Kaggle T4 with bf16/fp16.

In [ ]:
# texts_full, _, _ = build_windows(random_state=42)
# encoded_full = [encode(t) for t in texts_full]
# ds_full = Dataset.from_list(encoded_full).train_test_split(test_size=0.05, seed=42)
#
# full_out = REPO_DIR / 'models' / 'ner'
# args_full = TrainingArguments(
#     output_dir=str(full_out),
#     num_train_epochs=2,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     learning_rate=3e-5,
#     weight_decay=0.01,
#     warmup_ratio=0.1,
#     bf16=use_bf16,
#     fp16=use_fp16,
#     save_strategy='steps',
#     save_steps=2000,
#     eval_strategy='steps',
#     eval_steps=2000,
#     logging_steps=200,
#     save_total_limit=2,
#     report_to='none',
#     dataloader_num_workers=2,
#     seed=42,
#     metric_for_best_model='f1',
#     load_best_model_at_end=True,
# )
# trainer_full = Trainer(
#     model=AutoModelForTokenClassification.from_pretrained(
#         MODEL_ID, num_labels=len(LABEL_LIST), id2label=ID2LABEL, label2id=LABEL2ID,
#         ignore_mismatched_sizes=True,
#     ),
#     args=args_full,
#     data_collator=collator,
#     train_dataset=ds_full['train'],
#     eval_dataset=ds_full['test'],
#     processing_class=tokenizer,
#     compute_metrics=compute_metrics,
# )
# trainer_full.train()
# full_metrics = trainer_full.evaluate()
# trainer_full.save_model(str(full_out / 'final'))
# tokenizer.save_pretrained(str(full_out / 'final'))
# (full_out / 'final' / 'ner_metrics.json').write_text(json.dumps(full_metrics, indent=2))
# (full_out / 'final' / 'ner_label_map.json').write_text(json.dumps(ID2LABEL, indent=2))
# print('full eval:', full_metrics)

## 7. Inspect artifacts and export to ONNX

ONNX export uses `optimum` so the same artifact format that production NER inference expects is produced here. The export is over the picked checkpoint (full > sample) — copy to `models/ner/final/` if you only ran the sampled cell.

In [ ]:
from optimum.onnxruntime import ORTModelForTokenClassification

full_dir = REPO_DIR / 'models' / 'ner' / 'final'
sample_dir = REPO_DIR / 'models' / 'ner-sample' / 'final'
src_dir = full_dir if full_dir.exists() else sample_dir

if not src_dir.exists():
    raise FileNotFoundError('No NER artifact found. Run the sampled or full training cell first.')

print(f'=== {src_dir} ===')
for path in sorted(src_dir.glob('*')):
    if path.is_file():
        print(f'  {path.name}: {path.stat().st_size:,} bytes')

ort = ORTModelForTokenClassification.from_pretrained(src_dir, export=True)
ort.save_pretrained(src_dir)
print('ONNX exported into:', src_dir)
for path in sorted(src_dir.glob('*.onnx')):
    print(f'  {path.name}: {path.stat().st_size:,} bytes')

## 8. Package artifacts

Zip `models/ner/final/` into `/kaggle/working/` so it appears in Kaggle's notebook outputs.

In [ ]:
import shutil

tag = 'full' if (REPO_DIR / 'models' / 'ner' / 'final').exists() else 'sample'
src_dir = REPO_DIR / ('models/ner/final' if tag == 'full' else 'models/ner-sample/final')
out_zip = KAGGLE_WORKING / f'logfilter-ner-{tag}-artifacts'
shutil.make_archive(
    str(out_zip), 'zip',
    root_dir=src_dir.parent.parent,
    base_dir=Path(*src_dir.parts[-3:]).as_posix(),
)
print('packaged:', out_zip.with_suffix('.zip'))

## 9. Output description + how to consume in repo

Download `/kaggle/working/logfilter-ner-full-artifacts.zip` (or `-sample-` for the verification run) and extract it at the repository root. The archive contains:

```text
models/ner/final/config.json
models/ner/final/model.safetensors
models/ner/final/tokenizer.json (and tokenizer_config.json, special_tokens_map.json)
models/ner/final/model.onnx
models/ner/final/ner_metrics.json
models/ner/final/ner_label_map.json
```

To use this fine-tuned NER inside the production pipeline, point [src/logfilter/models/ner.py](../src/logfilter/models/ner.py) at the local directory:

```yaml
# config/config.yaml
models:
  ner:
    model_id: "models/ner/final"   # was: cisco-ai/SecureBERT2.0-NER
    device: "cpu"
    batch_size: 32
    min_confidence: 0.80
```

Caveats:

* **Weak supervision = noisy labels.** Treat the published per-entity F1 as a relative signal across runs, not as a calibrated absolute. The model can only learn what regex can label; it will not invent new entity classes.
* **Domain transfer test.** Before promoting this artifact to production, run inference on a held-out non-HDFS log corpus (a few thousand syslog/auth/nginx events) and inspect entity yield. If recall on `Indicator` is materially lower than regex on the same corpus, the fine-tune over-fitted to HDFS template syntax.
* **Composability with MLM pre-training.** For best results, run [kaggle_pretrain_mlm.ipynb](kaggle_pretrain_mlm.ipynb) first, then change `MODEL_ID` here to `models/securebert2-logs-mlm/final` so the encoder is log-adapted before the NER head is trained.